# 03. The limit

**Tier:** Deep. **Why you'd use this:** A deep tree walk can hit Python's recursion limit even when the last call looks like a "tail" call. Sometimes a loop over an explicit stack is the safer tool.

**After this lesson you can:** explain that a tail call still grows the stack in Python; rewrite a tree walk as a loop over a list.
About 25 minutes. **Uses:** [Nested data](02-structures-lesson.ipynb), [Errors](../06-errors/01-try-except-lesson.ipynb)


## Quick review

1. On a nested list `[1, [2, 3]]`, does the outer sum finish before the inner sum?
2. Why does recursion fit trees better than a single flat `for`?

<details><summary>Answers</summary>

No — the inner structure is summed during the outer call. A tree's children are the same kind of problem as the root.
</details>


## The idea

Each recursive call keeps a frame until it returns. Python caps how many frames a thread may hold. Past that cap, the call raises `RecursionError` instead of allocating frames until the machine dies. The cap is a safety rail, not the maximum your algorithm is allowed to need.

A tail call is a self-call that is the entire `return`. Nothing in the frame runs after it. Some languages reuse that frame. Python does not. The next frame is added anyway, and a long enough tail still raises.

An explicit stack is an ordinary list you push and pop yourself. It stores the same "what is left to visit" information a frame would have stored, without the cap of a thousand frames.

<details><summary>Explain more</summary>

The default cap is about a thousand frames, and the notebook is already using some of them before your function starts. A chain slightly under the cap can succeed. A chain of several thousand will not. Raising the cap with `sys.setrecursionlimit` asks for a bigger C stack too. If that stack is exhausted, the process can die with no Python traceback. Prefer a list.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

Those languages also keep frames for a call that is not optimized, and most of them do not optimize tail calls either. An explicit stack is the portable fix when the depth comes from the data.
</details>


## See it work

A chain of tuple nodes longer than the cap is still a small amount of data. Summing it recursively will not finish. Summing it with a list will. The list holds nodes that have not been added yet. `pop` takes the newest one, the way a call stack takes the newest frame. Children are pushed so they are visited next.


In [1]:
def tree_sum_stack(node: tuple) -> int:
    # 1. The root is waiting, the way a first call would be.
    pending = [node]
    total = 0
    while pending:
        # 2. Take the newest node.
        value, children = pending.pop()
        total += value
        # 3. Its children wait, newest pushed last so it is next.
        for child in children:
            pending.append(child)
    return total


sample = (2, ((1, ()), (4, ((3, ()),))))
print(tree_sum_stack(sample))


def make_chain(length: int) -> tuple:
    node: tuple = (1, ())
    for _ in range(length - 1):
        node = (1, (node,))
    return node


print(tree_sum_stack(make_chain(2000)))

10
2000


| step | popped value | pending values after the push |
| --- | --- | --- |
| start |  | 2 |
| 1 | 2 | 1, 4 |
| 2 | 4 | 1, 3 |
| 3 | 3 | 1 |
| 4 | 1 |  |

Pushing the left child first and the right child second means the right child is popped next. The total is still 10. Order of addition does not matter for a sum. It would matter if you were building a list of values. The chain of 2000 ones sums to 2000. A recursive `tree_sum` on that chain hits the cap. The next cell prints `RecursionError`.


In [2]:
def tree_sum(node: tuple) -> int:
    value, children = node
    total = value
    for child in children:
        total += tree_sum(child)
    return total


def make_chain(length: int) -> tuple:
    node: tuple = (1, ())
    for _ in range(length - 1):
        node = (1, (node,))
    return node


try:
    tree_sum(make_chain(2000))
except RecursionError as error:
    print(type(error).__name__)

RecursionError


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `return tail(n - 1)` still raise, if the frame has nothing left to do?
2. After the first pop on `sample`, which value is at the end of `pending`, and why?
3. Why is `chain(100)` fine when `chain(10000)` is not, even though both shrink `n`?

<details><summary>Answers</summary>

1. Python does not remove the current frame before the call. "Nothing left to do" is a fact about your source. The interpreter still stacks a frame per call so a traceback can list them.
2. The children are pushed left to right, so 1 is pushed, then 4. The end of the list is 4, and `pop` takes 4 next.
3. Both are correct recursive functions. Only the second needs more frames than the cap allows. Correctness and the cap are separate.
</details>


In [3]:
def chain(n: int) -> int:
    if n == 0:
        return 0
    return 1 + chain(n - 1)


print(chain(100))

100


In [4]:
import sys

print(sys.getrecursionlimit())

1000


In [5]:
def chain(n: int) -> int:
    if n == 0:
        return 0
    return 1 + chain(n - 1)


try:
    chain(10000)
except RecursionError as error:
    print(type(error).__name__)

RecursionError


<details><summary>Check your answers</summary>

- `chain(100)` returns 100. One hundred frames fit.
- `sys.getrecursionlimit()` prints the cap on this process. It is often 1000, and it can be changed, so treat the number as a fact about this run.
- The third cell prints `RecursionError`. `chain(10000)` hit the frame cap. The input was shrinking. There were just more steps than frames allowed. A raw traceback would repeat the self-call once per frame and put that name on the last line.
</details>


## Why it works this way

The cap exists so a runaway self-call fails inside Python. Without it, a function that never reached a base case would allocate frames until the operating system killed the process. You would get no `RecursionError` and often no traceback. The number you print with `sys.getrecursionlimit` counts frames, not data nodes. Your function shares that budget with the frames already above it. This is a CPython detail: the default is 1000, and a different build or a previous `sys.setrecursionlimit` can print another number. Do not write code that needs the cap to be a particular value.

A tail call looks like it could reuse one frame. `return chain(n - 1)` does nothing after the call except hand the result back. Python still builds a frame. The cell below is only that return, and on a long input it prints `RecursionError`. The language keeps the frame so every waiting call can appear in a traceback, and so the common case, a call with more work after it, has a place to resume. Two recursive calls in one function, the tree case, are not a tail call at all: after the first child returns, the loop still has to visit the next child. Eliminating tail calls would not make `tree_sum` constant-memory.

The list version stores nodes, not frames. A node is a tuple already in memory. Pushing it onto `pending` adds a reference. Two thousand references are ordinary list memory. Two thousand frames are not, because each frame holds the interpreter's bookkeeping and because the cap stops you first. Time stays proportional to the number of nodes: each node is popped once and each child is pushed once. The list's extra memory is proportional to the number of nodes waiting, which for a chain is the whole chain at the start of the walk only if you pushed them all up front. This walk pushes children when the parent is popped, so a chain holds one pending node at a time after the root is taken. A wide node can make `pending` long. That is the same "width versus depth" trade as the frame stack, with a higher ceiling.

```
frames:   [root] [child] [grandchild] ... cap
list:     pending = [next node, ...]    no frame cap
```

Raising the recursion limit is the wrong fix. It moves the crash from a catchable error to a possible hard abort, and it does not change the fact that a frame is a heavy way to store "visit this node next".


In [6]:
def tail(n: int) -> int:
    if n == 0:
        return 0
    return tail(n - 1)


try:
    tail(10000)
except RecursionError as error:
    print(type(error).__name__)

RecursionError


## When to use it

Keep the recursive form when the depth is small and you can see that it is small: a parsed expression, a tournament bracket, a tree you built with a balanced insert. The code matches the picture, and the cap is far away.

Switch to an explicit stack when the depth is up to the caller. A linked chain, a skewed tree, or a nested list from a file can be longer than a thousand. The list walk is the same algorithm with the stack written down. Use a plain `for` loop when there is no nesting at all. A stack of numbers you could have totaled with `sum` is extra machinery.

Do not call `sys.setrecursionlimit` to silence `RecursionError` in a program that reads outside data. Catching the error and reporting "this input is nested too deeply" is honest. Rewriting the walk is the repair.


## Common mistake

Popping the node and adding its value, then forgetting to push the children, sums only the root. The loop ends because `pending` is empty. The result is a number, so tests that only pass a leaf still pass.


In [7]:
def tree_sum_wrong(node: tuple) -> int:
    pending = [node]
    total = 0
    while pending:
        value = pending.pop()[0]
        total += value
    return total


sample = (2, ((1, ()), (4, ((3, ()),))))
print(tree_sum_wrong(sample))

2


The print is `2`. The pop kept index 0, the value, and dropped the children tuple, so 1, 4, and 3 never entered `pending`. The fix unpacks `children` and appends each one before the next pop.


In [8]:
def tree_sum_stack(node: tuple) -> int:
    pending = [node]
    total = 0
    while pending:
        value, children = pending.pop()
        total += value
        for child in children:
            pending.append(child)
    return total


sample = (2, ((1, ()), (4, ((3, ()),))))
print(tree_sum_stack(sample))

10


## Try it

Change `tree_sum_stack` so it returns the values in the order they are popped, not their sum. Then push the children from right to left instead of left to right. Which value is popped second for `sample`, and why did that order change when the push order changed?


## Recap

- `RecursionError` means the frame cap was hit, not that the base case is wrong.
- A tail call still allocates a frame. Python does not reuse it.
- A list you pop yourself can walk a chain that recursion will refuse.

Next: [practice](03-limit-practice.ipynb).
